# 13 — Mine hard positives from the existing cALorie evidence

Notebook 12 showed that the v5 gate rejects `unknown` motion well but rejects too many real supported exercises. This notebook finds those missed positives **inside the landmark archives we already built**:

- Workout/Fitness;
- MM-Fit exercise and `unknown` clips;
- all three labelled long-form YouTube batches;
- the Notebook 10 hard-negative/variant archive.

It does **not** download another multi-gigabyte dataset and does not rerun MediaPipe. It downloads roughly 108 MB of versioned landmark ZIPs and the frozen v5 checkpoint from this repository, reconstructs the exact group-separated v5 split, and scores supported windows from the **training split only**. Validation, internal test, and locked GainsByBrains remain untouched.

A supported training window with gate probability below the checkpoint's `0.90` threshold is a **hard positive**. The output `calorie_v5_hard_positive_mining.zip` contains ranked windows, clip summaries, recommended sampling weights, and pose previews. Notebook 14 will use the weights on the original windows; it must not append copied clips as if they were new independent data.

Run on **CPU**. This should take minutes rather than the long MediaPipe extraction runs.

In [ ]:
%pip install -q scikit-learn pandas==2.2.3 matplotlib

In [ ]:
from pathlib import Path
import json
import random
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from google.colab import files
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
WINDOW = 64
STRIDE = 32
MAX_HARD_WINDOWS_PER_CLASS = 120
MAX_PREVIEW_CLIPS_PER_CLASS = 4

# These three long-video groups are still marked draft/unreviewed by the
# repository annotation audit. New experiments must not mine or upweight them.
UNREVIEWED_GROUPS = {
    'youtube:oLsonHQJOUc',
    'youtube:c4TomZjzobY',
    'youtube:W2Bm9dfQjbk',
}

OUTPUT_ROOT = Path('/content/calorie_v5_hard_positive_mining')
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Runtime: CPU is enough')

## Load the same seven archives used by v5

The files come directly from the versioned project artifacts. `group_id` remains attached to every window. Related windows therefore retain the same train/validation/test assignment.

In [ ]:
REPOSITORY_RAW = (
    'https://raw.githubusercontent.com/bariscelikk1/cALorie/'
    'feat/tcn-rebuild/ml/artifacts'
)
LANDMARK_ARCHIVES = [
    ('v1', 'calorie_workoutfitness_landmarks_v1.zip'),
    ('v1', 'calorie_mmfit_landmarks_v1.zip'),
    ('v1', 'calorie_mmfit_unknown_v1.zip'),
    ('v2', 'calorie_youtube_mixed_landmarks_v3.zip'),
    ('v3', 'calorie_youtube_additional_landmarks_v1.zip'),
    ('v3', 'calorie_youtube_training_landmarks_v4.zip'),
    ('v5', 'calorie_workoutfitness_gate_hardneg_v1.zip'),
]


def download(url, destination):
    request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request, timeout=300) as response, destination.open('wb') as output:
        shutil.copyfileobj(response, output)
    print(f'{destination.name}: {destination.stat().st_size / 1e6:.1f} MB')


zip_paths = []
for version, name in LANDMARK_ARCHIVES:
    destination = Path('/content') / name
    download(f'{REPOSITORY_RAW}/{version}/landmarks/{name}', destination)
    zip_paths.append(destination)

MODEL_PATH = Path('/content/two_stage_tcn_v5.pt')
download(f'{REPOSITORY_RAW}/v5/model/two_stage_tcn.pt', MODEL_PATH)

SOURCE_ROOT = Path('/content/v5_landmark_sources')
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir()
for index, archive_path in enumerate(zip_paths):
    destination = SOURCE_ROOT / f'source_{index}'
    destination.mkdir()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)

frames = []
for manifest_path in SOURCE_ROOT.rglob('manifest.csv'):
    frame = pd.read_csv(manifest_path).fillna('')
    frame['npz_path'] = frame['output_path'].map(
        lambda value: str(manifest_path.parent / value)
    )
    frames.append(frame)

assert frames, 'No manifest.csv found in the versioned archives.'
manifest = pd.concat(frames, ignore_index=True)
manifest = manifest[
    (manifest.status == 'ok')
    & manifest.npz_path.map(lambda value: Path(value).exists())
].copy()
if 'group_id' not in manifest or manifest.group_id.eq('').any():
    manifest['group_id'] = manifest.apply(
        lambda row: f"{row['source_dataset']}:{row['clip_id']}", axis=1
    )

print('Valid clips:', len(manifest))
print('Groups:', manifest.group_id.nunique())
print('Unreviewed groups present but excluded from mining:',
      sorted(set(manifest.group_id) & UNREVIEWED_GROUPS))
display(manifest.canonical_class.value_counts().sort_index().rename('clips'))

## Reconstruct v5 windows and its immutable split

Each clip is resampled to 15 FPS when required. Windows contain 64 frames (about 4.3 seconds) with a stride of 32 frames. The same `StratifiedGroupKFold`, seed, ordering, and labels as Notebook 11 reconstruct the v5 train/validation/test assignment before any mining occurs.

In [ ]:
def resample_sequence(sequence, source_fps, target_fps=15):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([
        np.interp(new_x, old_x, flat[:, index])
        for index in range(flat.shape[1])
    ], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


windows, labels, groups, clip_ids, sources, starts, npz_paths = [], [], [], [], [], [], []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item['landmarks'].astype(np.float32)
    if sequence.ndim != 3 or sequence.shape[1:] != (17, 4):
        continue
    sequence = resample_sequence(sequence, getattr(row, 'source_fps', 15) or 15)
    clip_starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and clip_starts[-1] != len(sequence) - WINDOW:
        clip_starts.append(len(sequence) - WINDOW)
    for start in clip_starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([
                window, np.repeat(window[-1:], WINDOW - len(window), axis=0)
            ])
        windows.append(window)
        labels.append(row.canonical_class)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        sources.append(row.source_dataset)
        starts.append(start)
        npz_paths.append(row.npz_path)

X = np.asarray(windows, dtype=np.float32)
labels = np.asarray(labels)
groups = np.asarray(groups)
clip_ids = np.asarray(clip_ids)
sources = np.asarray(sources)
starts = np.asarray(starts, dtype=np.int64)
npz_paths = np.asarray(npz_paths)
ALL_CLASSES = sorted(np.unique(labels).tolist())
y_all = np.asarray([ALL_CLASSES.index(name) for name in labels], dtype=np.int64)

outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_val_idx, test_idx = next(outer.split(X, y_all, groups))
inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED + 1)
inner_train, inner_val = next(
    inner.split(X[train_val_idx], y_all[train_val_idx], groups[train_val_idx])
)
train_idx = train_val_idx[inner_train]
val_idx = train_val_idx[inner_val]

assert not (set(groups[train_idx]) & set(groups[val_idx]))
assert not (set(groups[train_idx]) & set(groups[test_idx]))
assert not (set(groups[val_idx]) & set(groups[test_idx]))
print('Windows:', X.shape)
for name, indices in [('train', train_idx), ('validation', val_idx), ('test', test_idx)]:
    print(name, 'windows=', len(indices), 'groups=', len(np.unique(groups[indices])))

## Score only reviewed supported windows from the training split

The gate architecture and weights are frozen. Mining cannot see validation, internal test, or GainsByBrains. Draft long-video groups detected by the new annotation audit are excluded even when their old landmark archive exists.

In [ ]:
class TemporalBlock(torch.nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = torch.nn.Sequential(
            torch.nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            torch.nn.BatchNorm1d(out_channels), torch.nn.ReLU(), torch.nn.Dropout(dropout),
            torch.nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            torch.nn.BatchNorm1d(out_channels), torch.nn.ReLU(), torch.nn.Dropout(dropout),
        )
        self.residual = (
            torch.nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else torch.nn.Identity()
        )

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(torch.nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = torch.nn.Sequential(*blocks)
        self.classifier = torch.nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))


checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
GATE_THRESHOLD = float(checkpoint['gate_threshold'])
gate_model = SmallTCN(int(checkpoint['input_features']), 2)
gate_model.load_state_dict(checkpoint['gate_state_dict'])
gate_model.eval()

training_supported = train_idx[
    (labels[train_idx] != 'unknown')
    & (~np.isin(groups[train_idx], list(UNREVIEWED_GROUPS)))
]
features = X[training_supported, :, :, :2].reshape(
    len(training_supported), WINDOW, -1
).transpose(0, 2, 1)
probability_parts = []
with torch.no_grad():
    for offset in range(0, len(features), 512):
        batch = torch.from_numpy(features[offset:offset + 512])
        probability_parts.append(gate_model(batch).softmax(1)[:, 1].numpy())
supported_probability = np.concatenate(probability_parts)

window_frame = pd.DataFrame({
    'window_key': [
        f'{source}|{group}|{clip}|{start}'
        for source, group, clip, start in zip(
            sources[training_supported], groups[training_supported],
            clip_ids[training_supported], starts[training_supported]
        )
    ],
    'source_dataset': sources[training_supported],
    'group_id': groups[training_supported],
    'clip_id': clip_ids[training_supported],
    'canonical_class': labels[training_supported],
    'window_start_frame': starts[training_supported],
    'window_end_frame': starts[training_supported] + WINDOW,
    'gate_supported_probability': supported_probability,
    'gate_prediction': np.where(
        supported_probability >= GATE_THRESHOLD, 'supported', 'unknown'
    ),
    'npz_path': npz_paths[training_supported],
})
window_frame['is_hard_positive'] = (
    window_frame.gate_supported_probability < GATE_THRESHOLD
)
window_frame['recommended_sample_weight'] = np.where(
    window_frame.is_hard_positive,
    1.0 + np.minimum(
        2.0,
        2.0 * (GATE_THRESHOLD - window_frame.gate_supported_probability) / GATE_THRESHOLD,
    ),
    1.0,
)

print('Frozen checkpoint:', checkpoint['experiment'])
print('Gate threshold:', GATE_THRESHOLD)
print('Reviewed supported training windows:', len(window_frame))
print('Hard-positive windows:', int(window_frame.is_hard_positive.sum()))
display(window_frame.groupby('canonical_class').agg(
    windows=('window_key', 'count'),
    hard_positives=('is_hard_positive', 'sum'),
    mean_probability=('gate_supported_probability', 'mean'),
    min_probability=('gate_supported_probability', 'min'),
).round(3))

## Select a balanced hard-positive curriculum

We rank rejected supported windows by gate probability within each class and cap the count per class. This prevents a large class from dominating Notebook 14. The recommended weight ranges from 1 to 3; it upweights original training windows without pretending they are new independent clips.

In [ ]:
hard_windows = window_frame[window_frame.is_hard_positive].copy()
selected_parts = []
for canonical, group in hard_windows.groupby('canonical_class'):
    selected_parts.append(
        group.sort_values('gate_supported_probability').head(MAX_HARD_WINDOWS_PER_CLASS)
    )
selected_windows = pd.concat(selected_parts, ignore_index=True) if selected_parts else hard_windows
selected_windows['selection_rank_in_class'] = selected_windows.groupby(
    'canonical_class'
).gate_supported_probability.rank(method='first').astype(int)

clip_summary = window_frame.groupby([
    'source_dataset', 'group_id', 'clip_id', 'canonical_class', 'npz_path'
]).agg(
    windows=('window_key', 'count'),
    hard_positive_windows=('is_hard_positive', 'sum'),
    mean_supported_probability=('gate_supported_probability', 'mean'),
    min_supported_probability=('gate_supported_probability', 'min'),
).reset_index()
clip_summary['hard_positive_fraction'] = (
    clip_summary.hard_positive_windows / clip_summary.windows
)
clip_summary['selected_windows'] = clip_summary.clip_id.map(
    selected_windows.clip_id.value_counts()
).fillna(0).astype(int)
clip_summary = clip_summary.sort_values([
    'hard_positive_fraction', 'mean_supported_probability'
], ascending=[False, True])

selected_windows.to_csv(OUTPUT_ROOT / 'hard_positive_windows.csv', index=False)
window_frame.drop(columns=['npz_path']).to_csv(
    OUTPUT_ROOT / 'all_reviewed_training_window_scores.csv', index=False
)
clip_summary.drop(columns=['npz_path']).to_csv(
    OUTPUT_ROOT / 'hard_positive_clips.csv', index=False
)

print('Balanced selected hard-positive windows:', len(selected_windows))
display(selected_windows.groupby('canonical_class').agg(
    selected=('window_key', 'count'),
    mean_probability=('gate_supported_probability', 'mean'),
    mean_weight=('recommended_sample_weight', 'mean'),
).round(3))

assert set(selected_windows.group_id).isdisjoint(set(groups[val_idx]))
assert set(selected_windows.group_id).isdisjoint(set(groups[test_idx]))
assert not (set(selected_windows.group_id) & UNREVIEWED_GROUPS)
assert selected_windows.gate_supported_probability.lt(GATE_THRESHOLD).all()

## Pose previews and audit package

The previews show six landmark frames from the hardest clips. They are model-input checks, not new label approval. Their purpose is to catch broken poses, heavy occlusion, or a window dominated by transition before changing sampling weights.

In [ ]:
SKELETON_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]
PREVIEW_ROOT = OUTPUT_ROOT / 'pose_previews'
PREVIEW_ROOT.mkdir()
preview_rows = []

for canonical, group in clip_summary[clip_summary.selected_windows > 0].groupby('canonical_class'):
    for row in group.head(MAX_PREVIEW_CLIPS_PER_CLASS).itertuples(index=False):
        with np.load(row.npz_path, allow_pickle=False) as item:
            sequence = item['landmarks'].astype(np.float32)
        indices = np.linspace(0, len(sequence) - 1, 6, dtype=int)
        fig, axes = plt.subplots(1, 6, figsize=(15, 3))
        for axis, frame_index in zip(axes, indices):
            xy = sequence[frame_index, :, :2]
            for left, right in SKELETON_EDGES:
                axis.plot(
                    [xy[left, 0], xy[right, 0]],
                    [-xy[left, 1], -xy[right, 1]],
                    color='tab:blue', linewidth=1.5,
                )
            axis.scatter(xy[:, 0], -xy[:, 1], s=12, color='tab:red')
            axis.set_aspect('equal')
            axis.axis('off')
        fig.suptitle(
            f'{canonical} | {row.clip_id} | hard={row.hard_positive_fraction:.0%} '
            f'| mean gate={row.mean_supported_probability:.3f}'
        )
        fig.tight_layout()
        preview_path = PREVIEW_ROOT / f'{canonical}__{row.clip_id}.png'
        fig.savefig(preview_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        preview_rows.append({
            'canonical_class': canonical,
            'clip_id': row.clip_id,
            'preview_path': str(preview_path.relative_to(OUTPUT_ROOT)),
        })

pd.DataFrame(preview_rows).to_csv(OUTPUT_ROOT / 'preview_index.csv', index=False)

summary = {
    'experiment': 'v5_existing_training_hard_positive_mining',
    'checkpoint': checkpoint['experiment'],
    'gate_threshold': GATE_THRESHOLD,
    'window_frames': WINDOW,
    'stride_frames': STRIDE,
    'source_archives': [path.name for path in zip_paths],
    'reviewed_supported_training_windows': int(len(window_frame)),
    'all_hard_positive_windows': int(window_frame.is_hard_positive.sum()),
    'balanced_selected_windows': int(len(selected_windows)),
    'selected_by_class': selected_windows.canonical_class.value_counts().sort_index().to_dict(),
    'excluded_unreviewed_groups': sorted(set(manifest.group_id) & UNREVIEWED_GROUPS),
    'validation_windows_used_for_mining': 0,
    'internal_test_windows_used_for_mining': 0,
    'locked_gainsbybrains_used': False,
    'creates_new_independent_evidence': False,
    'intended_use': 'sampling weights for original training windows in Notebook 14',
}
(OUTPUT_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2))
(OUTPUT_ROOT / 'README.txt').write_text(
    'Hard-positive mining report over existing v5 training evidence.\n'
    'Do not append these rows as new clips. Notebook 14 should apply the '
    'recommended weights to matching original window_key values.\n'
    'Validation, internal test, unreviewed draft groups, and locked GainsByBrains '
    'were excluded from mining.\n'
)
print(json.dumps(summary, indent=2))

zip_path = shutil.make_archive(
    '/content/calorie_v5_hard_positive_mining', 'zip', root_dir=OUTPUT_ROOT
)
print('ZIP ready:', zip_path)
print('ZIP size MB:', round(Path(zip_path).stat().st_size / 1024**2, 2))
files.download(zip_path)

## What comes next

Send back `calorie_v5_hard_positive_mining.zip`. Notebook 14 will keep the same two-stage architecture and original group assignments, exclude unresolved draft groups, and apply the selected weights only to matching training windows. Then a validation-only threshold will be selected again before the unchanged locked benchmark is run once in Notebook 15.